# Full Corpus Extraction

**Module 1 — Full run**

This notebook extracts text from **all** Enron email PDFs using a per-page tiered strategy:

1. **Try PyMuPDF `get_text()` first** on every page -- fast and accurate for PDFs with a text layer
2. **Fall back to built-in OCR** only on pages that return no text (image-only pages)

The output is one `.txt` file per PDF in `data/extracted_text/`, ready for parsing in Module 2.

## 1. Install Dependencies

In [14]:
%pip install pymupdf tqdm -q

Note: you may need to restart the kernel to use updated packages.


## 2. Imports & Configuration

In [15]:
import time
from pathlib import Path

import pymupdf
from tqdm.auto import tqdm

In [16]:
PDF_DIR = Path("enron_pdfs")
OUTPUT_DIR = Path("data/extracted_text")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files):,} PDFs in {PDF_DIR.resolve()}")
print(f"Output: {OUTPUT_DIR.resolve()}")

Found 4,911 PDFs in /Users/henryadamcollie/Documents/GitHub/1_pdf_to_kg_notebooks/enron_pdfs
Output: /Users/henryadamcollie/Documents/GitHub/1_pdf_to_kg_notebooks/data/extracted_text


## 3. Extraction function

Try `get_text()` on every page first -- it's near-instant and works on any file with a text layer. Only fall back to OCR on pages that return no text (image-only pages).

For the OCR fallback, we use PyMuPDF's built-in OCR rather than the Tesseract CLI. As we saw in notebook 1.2, both produce comparable results on clean B&W scans. The built-in method avoids subprocess overhead across 795 image-only files and keeps everything in a single dependency.


In [17]:
def extract_text(pdf_path):
    """Extract text from a PDF, using built-in OCR only for image-only pages.

    For each page, try PyMuPDF get_text() first (fast, accurate on text layers).
    If a page returns no text, fall back to PyMuPDF's built-in OCR for that page.

    Returns (text, page_count, method).
    method is 'text' (all pages had text), 'ocr' (all pages needed OCR),
    or 'mixed' (some pages text, some OCR).
    """
    doc = pymupdf.open(pdf_path)
    page_count = len(doc)
    pages = []
    ocr_pages = 0

    for page in doc:
        text = page.get_text().strip()
        if text:
            pages.append(text)
        else:
            # Image-only page — use PyMuPDF's built-in OCR (no subprocess)
            tp = page.get_textpage_ocr(dpi=300, language="eng")
            text = page.get_text(textpage=tp).strip()
            pages.append(text)
            ocr_pages += 1

    doc.close()

    if ocr_pages == 0:
        method = "text"
    elif ocr_pages == page_count:
        method = "ocr"
    else:
        method = "mixed"

    return "\n\n".join(pages), page_count, method

## 4. Extract all PDFs

Pre-extracted text files are included in the repo. If you'd like to run the extraction yourself (~20 minutes), uncomment the first line in the cell below to clear them and re-extract from scratch.

Digital files (small, no images) are processed first — these extract instantly. Scanned files (large, with images) come second and may trigger OCR on image-only pages, which is slower.


In [ ]:
# Uncomment the next two lines to clear existing files and re-extract from scratch
# for f in OUTPUT_DIR.glob("*.txt"):
#     f.unlink()

# Only extract files that don't already exist
to_extract = [p for p in pdf_files if not (OUTPUT_DIR / f"{p.stem}.txt").exists()]

if to_extract:
    print(f"Extracting {len(to_extract):,} files ({len(pdf_files) - len(to_extract):,} already exist)")
    
    results = []
    start = time.perf_counter()
    
    for pdf_path in tqdm(to_extract, desc="Extracting"):
        out_path = OUTPUT_DIR / f"{pdf_path.stem}.txt"
        try:
            text, page_count, method = extract_text(pdf_path)
            out_path.write_text(text, encoding="utf-8")
            results.append({
                "file": pdf_path.name,
                "pages": page_count,
                "characters": len(text),
                "method": method,
                "status": "ok" if text.strip() else "empty",
            })
        except Exception as e:
            results.append({
                "file": pdf_path.name,
                "pages": 0,
                "characters": 0,
                "method": "error",
                "status": f"error: {e}",
            })
    
    elapsed = time.perf_counter() - start
    print(f"\nExtracted {len(results):,} files in {elapsed:.0f}s")
else:
    # Build results from existing files
    print(f"All {len(pdf_files):,} files already extracted.")
    results = []
    for txt_path in sorted(OUTPUT_DIR.glob("*.txt")):
        text = txt_path.read_text(encoding="utf-8")
        results.append({
            "file": txt_path.name,
            "pages": 0,
            "characters": len(text),
            "method": "pre-extracted",
            "status": "ok" if text.strip() else "empty",
        })
    print(f"Loaded {len(results):,} results from existing files.")

Extracting 4,911 files (0 already exist)


Extracting: 100%|██████████| 4911/4911 [19:40<00:00,  4.16it/s]


Extracted 4,911 files in 1181s


## 5. Results Summary

In [22]:
text_count = sum(1 for r in results if r["method"] == "text")
ocr_count = sum(1 for r in results if r["method"] == "ocr")
mixed_count = sum(1 for r in results if r["method"] == "mixed")
ok_count = sum(1 for r in results if r["status"] == "ok")
empty_count = sum(1 for r in results if r["status"] == "empty")
error_count = sum(1 for r in results if r["status"].startswith("error"))
total_pages = sum(r["pages"] for r in results)
total_chars = sum(r["characters"] for r in results)

print(f"Total files:       {len(results):,}")
print(f"Successful:        {ok_count:,}")
print(f"Empty:             {empty_count:,}")
print(f"Errors:            {error_count:,}")
print()
print(f"Extraction method:")
print(f"  Text layer:      {text_count:,} files")
print(f"  Built-in OCR:    {ocr_count:,} files")
print(f"  Mixed:           {mixed_count:,} files")
print()
print(f"Total pages:       {total_pages:,}")
print(f"Total characters:  {total_chars:,}")
print(f"\nOutput directory:   {OUTPUT_DIR.resolve()}")

if error_count > 0:
    print(f"\n{error_count} problem files:")
    for r in results:
        if r["status"].startswith("error"):
            print(f"  {r['file']}: {r['status']}")

Total files:       4,911
Successful:        4,911
Empty:             0
Errors:            0

Extraction method:
  Text layer:      4,116 files
  Built-in OCR:    795 files
  Mixed:           0 files

Total pages:       7,316
Total characters:  10,880,514

Output directory:   /Users/henryadamcollie/Documents/GitHub/1_pdf_to_kg_notebooks/data/extracted_text


## 6. Verify Output

Spot-check a few extracted files to make sure they look reasonable.

In [23]:
txt_files = sorted(OUTPUT_DIR.glob("*.txt"))
print(f"{len(txt_files):,} .txt files in {OUTPUT_DIR}")
print()

# Show first 3 files
for txt_path in txt_files[:3]:
    text = txt_path.read_text(encoding="utf-8")
    print(f"=== {txt_path.name} ({len(text):,} chars) ===")
    print(text[:300])
    print()

4,911 .txt files in data/extracted_text

=== E0000813E.txt (944 chars) ===
CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0000813E
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From:
Rob Bradley <rob.bradley@enron.com>
Sent:
Tue, 21 Nov 2000 08:34:00 -0800 (PST)
To

=== E0000D1D0.txt (2,841 chars) ===
CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0000D1D0
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
CONFIDENTIAL - SUBJECT TO PROTECTIVE ORDER
From:
EDIS Email Service <edismail@incident.com>
Sent:
Fri, 5

=== E000B745B.txt (2,160 chars) ===
CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E000B745B
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From:
Susan Scott <susan.scott@enron.

### A note on your own data

Our dataset has a 44/40/16 split between digital, scanned, and image-only files. This is specific to our synthetic dataset — in a real document dump (like FOIA releases), every file is typically a scanned image with an OCR text layer. There are no "digital" PDFs because the documents were physically printed and scanned.

If the OCR text layers in your documents are good enough for your purposes, you can skip re-OCR entirely and extract with `get_text()` alone — instant, no dependencies, no configuration. Run a few samples through your parsing pipeline first to check whether the existing OCR quality is sufficient. If it is, you may not need OCR, Docling, or vision models at all.


## Done

All PDFs have been extracted to `data/extracted_text/`. Each `.txt` file contains the full text of the corresponding PDF.

**Next module:** We'll parse these text files into structured email records -- extracting From, To, Subject, date, and body fields.